#### 3. Shortest Path Analysis (Graph Algorithms)

Key Components

Data Reading: The script reads from a CSV file using pandas.

Graph Construction: It constructs a directed graph using networkx, where nodes represent airports and edges represent direct flights with weights as the actual flying time.

Path Finding: The code attempts to find the shortest paths from JFK to LAX based on the total flying time, printing the top three shortest paths.

In [2]:
import pandas as pd
import networkx as nx

In [5]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================

# Load dataset (each row = one flight)
df = pd.read_csv('August_2018_Nationwide.csv')

In [6]:
# ============================================================
# STEP 2: GRAPH CREATION
# ============================================================

# Create a DIRECTED graph
# WHY DIRECTED?
# Because flights are one-way:
# JFK → LAX is not same as LAX → JFK

G = nx.DiGraph()

In [7]:
# ============================================================
# STEP 3: ADD EDGES (BUILD GRAPH)
# ============================================================

# Each row = one flight
# We convert each flight into a graph edge

for idx, row in df.iterrows():

    # Only consider rows with valid flight time
    if pd.notna(row['ACTUAL_ELAPSED_TIME']):

        # Add edge from ORIGIN → DEST
        # weight = flying time (important for shortest path)
        G.add_edge(
            row['ORIGIN'],
            row['DEST'],
            weight=row['ACTUAL_ELAPSED_TIME']
        )

# RESULT:
# Graph is ready:
# Nodes = Airports
# Edges = Flights
# Weight = Flight time

In [8]:
# ============================================================
# STEP 4: DEFINE SOURCE AND DESTINATION
# ============================================================

airport_a = 'JFK'   # Source airport
airport_b = 'LAX'   # Destination airport

In [10]:
# ============================================================
# STEP 5: FIND SHORTEST PATHS
# ============================================================

# WHY shortest_simple_paths?
# → It generates multiple shortest paths in order
# → Internally uses Dijkstra-like logic

try:
    path_generator = nx.shortest_simple_paths(
        G,
        source=airport_a,
        target=airport_b,
        weight='weight'
    )

    shortest_time = None  # will store best (minimum) time

    # Fetch top 3 shortest paths
    for i in range(3):

        # Get next shortest path
        path = next(path_generator)

        # Calculate total time of this path
        total_time = sum(
            G[u][v]['weight']
            for u, v in zip(path[:-1], path[1:])
        )

        # First path = best (shortest)
        if i == 0:
            shortest_time = total_time
        else:
            # Compare with best path
            time_difference = total_time - shortest_time
            print(f"Difference from shortest path: {time_difference} minutes")

        print(f"Path {i+1}: {path}")
        print("Total time:", total_time, "minutes")
        # ============================================================
# STEP 6: ERROR HANDLING
# ============================================================

except StopIteration:
    # If less than 3 paths exist
    print("Less than three paths found")

except nx.NetworkXNoPath:
    # If no route exists
    print("No path found between airports")

Path 1: ['JFK', 'LAX']
Total time: 354.0 minutes
Difference from shortest path: 30.0 minutes
Path 2: ['JFK', 'IND', 'LAX']
Total time: 384.0 minutes
Difference from shortest path: 30.0 minutes
Path 3: ['JFK', 'CLE', 'LAX']
Total time: 384.0 minutes


Direct Efficiency: The direct flight from JFK to LAX is the most time-efficient route, taking 354 minutes.

Common Time Increase: Both alternative routes via IND and CLE increase travel time by exactly 30 minutes compared to the direct flight.

Routing Variability: Despite different midpoints (IND and CLE), the total travel time for the alternative routes remains identical, highlighting a possible consistency in layover or flight speeds for these paths.

Optimal Choice: For time-sensitive travel, choosing the direct route between JFK and LAX is clearly the best option.

Layover Impact: The layovers at IND and CLE have an equal impact on travel time, suggesting that either layover might be due to scheduling rather than distance.